In [3]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [5]:
ferrao_query = f"""select
    date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
    customerid,
    sum(ao_session) ao_session,
    sum(fe_session) fe_session,
    sum(rr_session) rr_session,
    sum(ao_day) ao_day,
    sum(fe_day) fe_day,
    sum(rr_day) rr_day
from
(
    select 
        day,
        customerid, 
        max(ao_sessions_unique_daily) ao_session, 
        max(fe_sessions_unique_daily) fe_session, 
        sum(rr_sessions_unique_daily) rr_session, 
        max(case when ao_sessions_unique_daily > 0 then 1 else 0 end) ao_day,
        max(case when fe_sessions_unique_daily > 0 then 1 else 0 end) fe_day,
        max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_day
    from 
        datasets.customer_rf_daily_kpi
    where 
        day >= '2024-12-16'
        and day <= '2025-03-16'
        and city = 'Bangalore'
        and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    order by 1
)
group by 1,2"""

In [5]:
# ferrao_df = pd.read_sql(ferrao_query, conn3)

In [7]:
# ferrao_df.to_csv('ferrao_df.csv', index=False, header=False)

In [3]:
# ferrao_df.columns

In [9]:
# ferrao_df

### new segment

In [36]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        segment,
        /*
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.65 ) w13_ao_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.75 ) w13_ao_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.90 ) w13_ao_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.95 ) w13_ao_ad_p95,

        /*
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.65 ) w13_fe_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.75 ) w13_fe_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.90 ) w13_fe_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.95 ) w13_fe_ad_p95,
        
        /*
        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.65 ) w13_rr_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.75 ) w13_rr_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.90 ) w13_rr_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.95 ) w13_rr_ad_p95

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1-DAILY,4.889526,5.232211,6.000000,6.941847,7.000000,4.935048,5.249039,5.997559,6.957982,7.000000,4.482966,5.004695,5.906038,6.372636,7.000000
1,2-WEEKLY,2.030823,2.998192,3.219209,4.432441,5.000000,2.042800,2.986881,3.149663,4.512393,5.008554,2.000000,2.488202,3.000318,4.000145,4.981408
2,3-MONTHLY,1.035551,1.981144,2.004736,3.022409,4.002926,1.046550,1.977786,2.000126,3.013962,4.000381,1.000000,1.090540,1.887431,2.802586,3.379108
3,4-OTHER,1.000000,1.486401,2.000000,3.000683,3.986819,1.000000,1.550370,1.999870,2.998963,3.994902,1.000000,1.000000,1.108887,2.059811,3.066617
4,5-INACTIVE,0.967203,1.000000,1.000000,1.010641,2.000000,0.981687,1.000000,1.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [23]:
cx_data0

,segment,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1-DAILY,5.372035,6.000000,6.004291,7.000000,7.000000,5.421511,6.000000,6.012614,7.000000,7.000000,5.060882,5.997901,6.000000,7.000000,7.000000
1,2-WEEKLY,2.512380,3.075144,3.965013,4.999862,5.937952,2.532005,3.079991,3.973778,5.000000,5.971343,2.025633,2.975930,3.378772,4.916590,5.224236
2,3-MONTHLY,1.055164,1.939252,2.000303,3.033583,4.008421,1.064242,1.965914,2.009735,3.033101,4.013538,1.000000,1.082092,1.928826,2.826521,3.487681
3,4-OTHER,1.000000,1.456299,2.000000,2.999637,3.998271,1.000000,1.539887,2.000000,3.000052,3.990021,1.000000,1.000000,1.115687,2.069415,3.020937
4,5-INACTIVE,0.964279,1.000000,1.000000,1.000000,2.000000,0.991013,1.000000,1.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [37]:
cx_data0.to_clipboard(index=False)

### Office bins

In [5]:
cx_data15 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
    
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        office_bin,
        /*
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.65 ) w13_ao_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.75 ) w13_ao_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.90 ) w13_ao_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.95 ) w13_ao_ad_p95,

        /*
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.65 ) w13_fe_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.75 ) w13_fe_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.90 ) w13_fe_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.95 ) w13_fe_ad_p95,
        
        /*
        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.65 ) w13_rr_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.75 ) w13_rr_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.90 ) w13_rr_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.95 ) w13_rr_ad_p95

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,a. no-office-transit,1.000003,1.371005,1.994067,2.999422,3.942463,1.000000,1.420931,1.999958,2.998994,3.951975,1.000000,1.000025,1.108073,2.098211,3.056626
1,b. office ride,1.974708,2.349240,3.013341,4.816862,5.663122,1.996614,2.386060,3.022661,4.772761,5.759572,1.293042,2.011999,2.757242,4.188784,5.181930
2,c. transit ride,1.040347,1.998622,2.033585,3.572983,4.621962,1.120113,1.996285,2.113715,3.619982,4.663938,1.000000,1.193979,1.982625,3.014963,4.029134


In [9]:
cx_data15

,office_bin,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,a. no-office-transit,1.000003,1.371005,1.994067,2.999422,3.942463,1.000000,1.420931,1.999958,2.998994,3.951975,1.000000,1.000025,1.108073,2.098211,3.056626
1,b. office ride,1.974708,2.349240,3.013341,4.816862,5.663122,1.996614,2.386060,3.022661,4.772761,5.759572,1.293042,2.011999,2.757242,4.188784,5.181930
2,c. transit ride,1.040347,1.998622,2.033585,3.572983,4.621962,1.120113,1.996285,2.113715,3.619982,4.663938,1.000000,1.193979,1.982625,3.014963,4.029134


In [10]:
cx_data15.to_clipboard(index=False)

In [7]:
cx_data16 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        segment,
        office_bin,
        /*
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.65 ) w13_ao_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.75 ) w13_ao_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.90 ) w13_ao_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.95 ) w13_ao_ad_p95,

        /*
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.65 ) w13_fe_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.75 ) w13_fe_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.90 ) w13_fe_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.95 ) w13_fe_ad_p95,
        
        /*
        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.65 ) w13_rr_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.75 ) w13_rr_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.90 ) w13_rr_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.95 ) w13_rr_ad_p95

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1-DAILY,a. no-office-transit,4.998262,5.037262,6.000000,7.000000,7.0,4.997324,5.052679,6.000000,6.956617,7.000000,4.552229,5.000000,5.997602,6.018246,7.000000
1,1-DAILY,b. office ride,4.938293,5.167971,5.997340,6.966792,7.0,4.945257,5.161344,5.999828,6.952001,7.000000,4.456113,5.003288,5.943415,6.371115,7.000000
2,1-DAILY,c. transit ride,5.000000,5.306889,6.000000,7.000000,7.0,5.000000,5.550605,6.000000,7.000000,7.000000,4.876407,5.000000,5.999682,6.429490,7.000000
3,2-WEEKLY,a. no-office-transit,2.000000,2.928286,3.000000,4.026258,5.0,2.000000,2.952831,3.009174,4.040305,5.000000,2.000000,2.044382,2.993204,4.000000,4.990530
4,2-WEEKLY,b. office ride,2.083700,2.999500,3.325556,4.553530,5.0,2.071994,2.999533,3.307292,4.624743,5.005333,1.999987,2.689665,3.000109,4.006136,4.995558


In [8]:
cx_data16

,segment,office_bin,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1-DAILY,a. no-office-transit,4.998262,5.037262,6.000000,7.000000,7.000000,4.997324,5.052679,6.000000,6.956617,7.000000,4.552229,5.000000,5.997602,6.018246,7.000000
1,1-DAILY,b. office ride,4.938293,5.167971,5.997340,6.966792,7.000000,4.945257,5.161344,5.999828,6.952001,7.000000,4.456113,5.003288,5.943415,6.371115,7.000000
2,1-DAILY,c. transit ride,5.000000,5.306889,6.000000,7.000000,7.000000,5.000000,5.550605,6.000000,7.000000,7.000000,4.876407,5.000000,5.999682,6.429490,7.000000
3,2-WEEKLY,a. no-office-transit,2.000000,2.928286,3.000000,4.026258,5.000000,2.000000,2.952831,3.009174,4.040305,5.000000,2.000000,2.044382,2.993204,4.000000,4.990530
4,2-WEEKLY,b. office ride,2.083700,2.999500,3.325556,4.553530,5.000000,2.071994,2.999533,3.307292,4.624743,5.005333,1.999987,2.689665,3.000109,4.006136,4.995558
5,2-WEEKLY,c. transit ride,2.000097,2.985082,3.015402,4.029676,5.000000,2.000000,2.986312,3.016718,4.035667,5.000000,2.000000,2.074389,2.999036,4.000000,4.999972
6,3-MONTHLY,a. no-office-transit,1.000000,1.590461,1.999751,2.996093,3.481539,1.000079,1.602278,2.000000,2.987964,3.543186,1.000000,1.000000,1.131063,2.000000,2.999327
7,3-MONTHLY,b. office ride,1.309588,2.000015,2.128681,3.507069,4.197172,1.351631,1.999990,2.197439,3.557250,4.266380,1.000018,1.605563,2.000000,2.999995,3.983195
8,3-MONTHLY,c. transit ride,1.000522,1.976133,2.000054,3.000000,3.971459,1.000619,1.977673,2.000000,3.000000,3.974738,1.000000,1.000077,1.815104,2.308155,3.003345
9,4-OTHER,a. no-office-transit,1.000000,1.080012,1.993109,2.790405,3.114614,1.000000,1.153188,1.973817,2.791252,3.207510,1.000000,1.000000,1.000185,2.000000,2.813758


In [ ]:
cx_data16.to_clipboard(index=False)

In [7]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2 
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,NRHA,5.390229,5.000000,5.000000,6.000000,5.044368,6.000000,6.000000,6.000000,6.000000,6.000000,5.750783,6.000000,5.833789,5.499857,5.000000,5.000000,6.000000,5.068210,6.000000,6.000000,6.000000,6.000000,6.000000,5.752702,6.000000,5.967399,5.020181,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.128714,6.000000,5.198168
1,1-DAILY,RHA,5.349086,4.993887,5.000000,5.941706,5.000000,6.000000,6.000000,5.999445,5.999255,5.996072,5.305899,5.964160,5.322025,5.422398,4.971036,5.000000,5.968670,5.000000,6.000000,6.000000,5.999615,6.000000,5.994234,5.409411,5.952671,5.428900,5.033700,4.440269,4.975274,5.377037,5.000000,5.794656,5.859682,5.843010,5.849201,5.509905,5.031787,5.382990,5.011753
2,1-DAILY,UNKNOWN,5.038358,4.962302,4.987978,5.970336,5.000000,5.966018,6.000000,5.986724,5.998694,5.987385,5.265818,5.968784,5.037728,5.106908,4.965394,4.998508,5.974879,5.000000,5.979816,6.000000,5.994066,6.000000,5.986033,5.385603,5.972664,5.056516,5.000000,4.073899,4.483251,5.302400,5.000000,5.599373,5.951034,5.794531,5.954343,5.586117,5.014155,5.443606,5.000000
3,2-WEEKLY,NRHA,2.016547,2.000000,2.006722,2.172732,2.017030,2.200948,2.044415,2.413537,2.367006,2.156208,2.020605,2.111704,2.064706,2.014346,2.000000,2.008204,2.187497,2.017877,2.239730,2.071518,2.454656,2.414974,2.170720,2.107261,2.132883,2.085943,2.000000,1.993668,2.000000,2.000000,2.000000,2.000000,2.000000,2.000462,2.000000,2.000000,2.000000,2.000000,2.000000
4,2-WEEKLY,RHA,2.280350,2.002264,2.024550,2.649452,2.285372,2.731144,2.666940,2.912127,2.879894,2.707419,2.562991,2.722094,2.570730,2.376463,2.005970,2.021923,2.667537,2.349452,2.829539,2.760342,2.914783,2.916536,2.736536,2.625406,2.739659,2.596767,2.001530,1.995916,1.998838,2.032981,2.002531,2.073711,2.030649,2.123749,2.164406,2.027340,2.006909,2.038051,2.023416


In [8]:
cx_data11

,segment,RHA_Check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,NRHA,5.390229,5.000000,5.000000,6.000000,5.044368,6.000000,6.000000,6.000000,6.000000,6.000000,5.750783,6.000000,5.833789,5.499857,5.000000,5.000000,6.000000,5.068210,6.000000,6.000000,6.000000,6.000000,6.000000,5.752702,6.000000,5.967399,5.020181,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.128714,6.000000,5.198168
1,1-DAILY,RHA,5.349086,4.993887,5.000000,5.941706,5.000000,6.000000,6.000000,5.999445,5.999255,5.996072,5.305899,5.964160,5.322025,5.422398,4.971036,5.000000,5.968670,5.000000,6.000000,6.000000,5.999615,6.000000,5.994234,5.409411,5.952671,5.428900,5.033700,4.440269,4.975274,5.377037,5.000000,5.794656,5.859682,5.843010,5.849201,5.509905,5.031787,5.382990,5.011753
2,1-DAILY,UNKNOWN,5.038358,4.962302,4.987978,5.970336,5.000000,5.966018,6.000000,5.986724,5.998694,5.987385,5.265818,5.968784,5.037728,5.106908,4.965394,4.998508,5.974879,5.000000,5.979816,6.000000,5.994066,6.000000,5.986033,5.385603,5.972664,5.056516,5.000000,4.073899,4.483251,5.302400,5.000000,5.599373,5.951034,5.794531,5.954343,5.586117,5.014155,5.443606,5.000000
3,2-WEEKLY,NRHA,2.016547,2.000000,2.006722,2.172732,2.017030,2.200948,2.044415,2.413537,2.367006,2.156208,2.020605,2.111704,2.064706,2.014346,2.000000,2.008204,2.187497,2.017877,2.239730,2.071518,2.454656,2.414974,2.170720,2.107261,2.132883,2.085943,2.000000,1.993668,2.000000,2.000000,2.000000,2.000000,2.000000,2.000462,2.000000,2.000000,2.000000,2.000000,2.000000
4,2-WEEKLY,RHA,2.280350,2.002264,2.024550,2.649452,2.285372,2.731144,2.666940,2.912127,2.879894,2.707419,2.562991,2.722094,2.570730,2.376463,2.005970,2.021923,2.667537,2.349452,2.829539,2.760342,2.914783,2.916536,2.736536,2.625406,2.739659,2.596767,2.001530,1.995916,1.998838,2.032981,2.002531,2.073711,2.030649,2.123749,2.164406,2.027340,2.006909,2.038051,2.023416
5,2-WEEKLY,UNKNOWN,2.477742,2.000000,2.016718,2.853613,2.396894,2.931510,2.868936,2.987555,2.979986,2.907810,2.839249,2.948279,2.773544,2.521006,2.004366,2.020055,2.851517,2.457970,2.952547,2.910264,2.973230,2.991338,2.923543,2.810914,2.961762,2.797617,2.006957,2.000000,2.000000,2.120288,2.018734,2.296461,2.217056,2.434196,2.564686,2.164949,2.087694,2.314025,2.065655
6,3-MONTHLY,NRHA,1.000412,1.000058,1.000000,1.002586,1.000042,1.000504,1.000000,1.000951,1.002320,1.001629,1.000809,1.000197,1.007715,1.000000,1.000408,1.000000,1.000857,1.000120,1.000000,1.000762,1.002310,1.004226,1.001203,1.000766,1.002382,1.001109,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,3-MONTHLY,RHA,1.002977,1.000656,1.000977,1.002159,1.006794,1.002417,1.015327,1.025221,1.051262,1.025699,1.012112,1.014217,1.036952,1.002549,1.000200,1.000613,1.004622,1.012666,1.023835,1.002521,1.063307,1.072034,1.025478,1.016112,1.016120,1.035484,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,3-MONTHLY,UNKNOWN,1.009972,1.000156,1.000000,1.004483,1.000359,1.029681,1.007874,1.086059,1.137781,1.039159,1.022620,1.060597,1.042422,1.009619,1.000222,1.000089,1.009698,1.000861,1.024113,1.031264,1.082219,1.121644,1.047027,1.048088,1.091511,1.069100,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,4-OTHER,NRHA,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0

In [11]:
cx_data11.to_clipboard(index=False)

In [9]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,Non-Zwigato,5.771312,5.000000,5.028978,6.000000,5.010538,6.000000,6.000000,6.000000,6.000000,6.000000,5.870958,6.000000,5.845604,5.627563,5.000000,5.025649,6.000000,5.062016,6.000000,6.000000,6.000000,6.000000,6.000000,5.961073,6.000000,5.954404,5.053545,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.044397,5.994986,5.061441
1,1-DAILY,UNKNOWN,5.134718,4.964513,5.000000,5.967585,5.000000,5.976685,6.000000,5.982265,5.999135,5.984704,5.271927,5.956300,5.030905,5.192411,4.959628,4.990533,5.980681,5.000000,5.974762,6.000000,5.985731,5.993305,5.981911,5.389743,5.975928,5.040268,5.000000,4.148826,4.448074,5.320225,5.000000,5.529246,5.967396,5.733024,5.960813,5.607786,5.026953,5.484875,5.000000
2,1-DAILY,Zwigato,5.250747,4.899959,5.000000,5.906313,5.000202,6.000000,6.000000,5.999270,6.000000,5.992156,5.243726,5.907432,5.280796,5.335955,4.947288,5.000000,5.922822,5.000000,6.000000,6.000000,6.000000,6.000000,5.990626,5.323462,5.925143,5.356605,5.039947,4.377531,4.969894,5.314565,5.000000,5.791241,5.864083,5.817070,5.900628,5.514004,5.015623,5.402849,5.000000
3,2-WEEKLY,Non-Zwigato,2.028333,2.003554,2.011230,2.364373,2.188111,2.454737,2.401090,2.648528,2.633265,2.376057,2.239986,2.345458,2.236690,2.052935,2.006506,2.006643,2.353906,2.200798,2.486571,2.426775,2.680740,2.672840,2.422826,2.277108,2.362942,2.195427,2.000000,1.993494,2.000000,2.000000,2.000000,2.000000,2.001197,2.000496,2.007465,2.000000,2.000000,2.000000,2.000000
4,2-WEEKLY,UNKNOWN,2.480300,2.002715,2.014795,2.901947,2.426653,2.930690,2.893839,2.970872,2.978331,2.921366,2.737115,2.930952,2.713922,2.518411,2.003453,2.016574,2.905772,2.452352,2.950378,2.921074,2.971949,2.982620,2.920481,2.849183,2.939050,2.836458,2.002509,1.998158,2.000000,2.099476,2.013488,2.304921,2.206929,2.434560,2.555243,2.251553,2.084908,2.323728,2.104871


In [10]:
cx_data12

,segment,Zwigato_check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,Non-Zwigato,5.771312,5.000000,5.028978,6.000000,5.010538,6.000000,6.000000,6.000000,6.000000,6.000000,5.870958,6.000000,5.845604,5.627563,5.000000,5.025649,6.000000,5.062016,6.000000,6.000000,6.000000,6.000000,6.000000,5.961073,6.000000,5.954404,5.053545,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.044397,5.994986,5.061441
1,1-DAILY,UNKNOWN,5.134718,4.964513,5.000000,5.967585,5.000000,5.976685,6.000000,5.982265,5.999135,5.984704,5.271927,5.956300,5.030905,5.192411,4.959628,4.990533,5.980681,5.000000,5.974762,6.000000,5.985731,5.993305,5.981911,5.389743,5.975928,5.040268,5.000000,4.148826,4.448074,5.320225,5.000000,5.529246,5.967396,5.733024,5.960813,5.607786,5.026953,5.484875,5.000000
2,1-DAILY,Zwigato,5.250747,4.899959,5.000000,5.906313,5.000202,6.000000,6.000000,5.999270,6.000000,5.992156,5.243726,5.907432,5.280796,5.335955,4.947288,5.000000,5.922822,5.000000,6.000000,6.000000,6.000000,6.000000,5.990626,5.323462,5.925143,5.356605,5.039947,4.377531,4.969894,5.314565,5.000000,5.791241,5.864083,5.817070,5.900628,5.514004,5.015623,5.402849,5.000000
3,2-WEEKLY,Non-Zwigato,2.028333,2.003554,2.011230,2.364373,2.188111,2.454737,2.401090,2.648528,2.633265,2.376057,2.239986,2.345458,2.236690,2.052935,2.006506,2.006643,2.353906,2.200798,2.486571,2.426775,2.680740,2.672840,2.422826,2.277108,2.362942,2.195427,2.000000,1.993494,2.000000,2.000000,2.000000,2.000000,2.001197,2.000496,2.007465,2.000000,2.000000,2.000000,2.000000
4,2-WEEKLY,UNKNOWN,2.480300,2.002715,2.014795,2.901947,2.426653,2.930690,2.893839,2.970872,2.978331,2.921366,2.737115,2.930952,2.713922,2.518411,2.003453,2.016574,2.905772,2.452352,2.950378,2.921074,2.971949,2.982620,2.920481,2.849183,2.939050,2.836458,2.002509,1.998158,2.000000,2.099476,2.013488,2.304921,2.206929,2.434560,2.555243,2.251553,2.084908,2.323728,2.104871
5,2-WEEKLY,Zwigato,2.233511,2.006168,2.015612,2.626776,2.266943,2.693392,2.598255,2.814369,2.860382,2.676273,2.518316,2.723159,2.533493,2.300943,2.000325,2.004591,2.630228,2.307952,2.708088,2.678056,2.829697,2.848175,2.709794,2.576234,2.731602,2.568528,2.000000,1.985255,1.998484,2.005035,2.000000,2.048751,2.016862,2.100001,2.167864,2.038642,2.018606,2.084150,2.023624
6,3-MONTHLY,Non-Zwigato,1.002424,1.000685,1.000365,1.007017,1.001361,1.004547,1.004403,1.004284,1.007756,1.000863,1.000132,1.000671,1.009138,1.001877,1.000992,1.000330,1.002029,1.001253,1.001017,1.002239,1.010738,1.005127,1.006526,1.000092,1.000935,1.002589,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,3-MONTHLY,UNKNOWN,1.001415,1.000000,1.000390,1.005544,1.004719,1.014181,1.004059,1.059645,1.076418,1.032491,1.040296,1.079061,1.064415,1.011405,1.000000,1.000339,1.002839,1.005222,1.017769,1.016987,1.095474,1.131688,1.056702,1.051659,1.093436,1.079927,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,3-MONTHLY,Zwigato,1.008811,1.000391,1.000413,1.010525,1.000774,1.002156,1.000716,1.045952,1.027588,1.017336,1.007290,1.031574,1.038531,1.015955,1.001494,1.000472,1.001254,1.000739,1.010785,1.003399,1.044072,1.065309,1.024204,1.021931,1.023661,1.051312,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,4-OTHER,Non-Zwigato,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00

In [12]:
cx_data12.to_clipboard(index=False)

In [13]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,1. BOTH,5.270684,4.904677,5.000000,5.913085,5.000000,5.994400,6.0,5.994254,6.000000,5.947041,5.231060,5.901241,5.252620,5.347537,4.929719,5.000000,5.934341,5.000000,6.000000,6.0,6.000000,6.0,5.981895,5.303852,5.930237,5.340330,5.007207,4.237827,4.954901,5.237532,5.0,5.762986,5.882309,5.830205,5.834308,5.457419,5.005235,5.399980,5.029572
1,1-DAILY,2. ONLY Zwigato,5.280684,5.000000,5.000000,6.000000,5.004487,6.000000,6.0,6.000000,6.000000,6.000000,5.576179,6.000000,5.487127,5.520235,5.000000,5.000000,6.000000,5.001857,6.000000,6.0,6.000000,6.0,6.000000,5.787877,6.000000,5.650951,5.037756,5.000000,5.000000,5.895955,5.0,6.000000,6.000000,6.000000,6.000000,6.000000,5.036550,6.000000,5.016511
2,1-DAILY,3. ONLY RHA,5.821422,5.000000,5.008906,6.000000,5.011427,6.000000,6.0,6.000000,6.000000,6.000000,6.000000,6.000000,5.987189,5.933906,5.000000,5.012012,6.000000,5.025592,6.000000,6.0,6.000000,6.0,6.000000,6.000000,6.000000,6.000000,5.011910,5.000000,5.000000,6.000000,5.0,6.000000,6.000000,6.000000,6.000000,6.000000,5.021193,5.768918,5.003221
3,1-DAILY,4. NONE,5.486086,5.000000,5.020791,6.000000,5.018828,6.000000,6.0,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.666099,5.000000,5.002577,6.000000,5.007337,6.000000,6.0,6.000000,6.0,6.000000,6.000000,6.000000,6.000000,5.012118,5.000000,5.000000,6.000000,5.0,6.000000,6.000000,6.000000,6.000000,6.000000,5.260930,6.000000,5.469425
4,1-DAILY,5. IOS,5.050008,4.957078,4.990143,5.966334,5.000000,5.971241,6.0,5.987168,5.997179,5.979425,5.282137,5.964113,5.073598,5.097697,4.964217,4.996986,5.973708,5.000000,5.984847,6.0,5.989715,6.0,5.984665,5.429348,5.977885,5.090357,5.000000,4.108587,4.465461,5.295317,5.0,5.598374,5.953362,5.732131,5.951964,5.578577,5.024273,5.457256,5.000000


In [14]:
cx_data13

,segment,tech_savvy,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,1-DAILY,1. BOTH,5.270684,4.904677,5.000000,5.913085,5.000000,5.994400,6.000000,5.994254,6.000000,5.947041,5.231060,5.901241,5.252620,5.347537,4.929719,5.000000,5.934341,5.000000,6.000000,6.000000,6.000000,6.000000,5.981895,5.303852,5.930237,5.340330,5.007207,4.237827,4.954901,5.237532,5.000000,5.762986,5.882309,5.830205,5.834308,5.457419,5.005235,5.399980,5.029572
1,1-DAILY,2. ONLY Zwigato,5.280684,5.000000,5.000000,6.000000,5.004487,6.000000,6.000000,6.000000,6.000000,6.000000,5.576179,6.000000,5.487127,5.520235,5.000000,5.000000,6.000000,5.001857,6.000000,6.000000,6.000000,6.000000,6.000000,5.787877,6.000000,5.650951,5.037756,5.000000,5.000000,5.895955,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.036550,6.000000,5.016511
2,1-DAILY,3. ONLY RHA,5.821422,5.000000,5.008906,6.000000,5.011427,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.987189,5.933906,5.000000,5.012012,6.000000,5.025592,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.011910,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.021193,5.768918,5.003221
3,1-DAILY,4. NONE,5.486086,5.000000,5.020791,6.000000,5.018828,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.666099,5.000000,5.002577,6.000000,5.007337,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.012118,5.000000,5.000000,6.000000,5.000000,6.000000,6.000000,6.000000,6.000000,6.000000,5.260930,6.000000,5.469425
4,1-DAILY,5. IOS,5.050008,4.957078,4.990143,5.966334,5.000000,5.971241,6.000000,5.987168,5.997179,5.979425,5.282137,5.964113,5.073598,5.097697,4.964217,4.996986,5.973708,5.000000,5.984847,6.000000,5.989715,6.000000,5.984665,5.429348,5.977885,5.090357,5.000000,4.108587,4.465461,5.295317,5.000000,5.598374,5.953362,5.732131,5.951964,5.578577,5.024273,5.457256,5.000000
5,2-WEEKLY,1. BOTH,2.308214,2.000662,2.023331,2.663149,2.315219,2.771075,2.704120,2.939447,2.944874,2.758573,2.578601,2.769338,2.601049,2.366810,2.001156,2.004154,2.698341,2.340024,2.806735,2.737513,2.901346,2.931807,2.815227,2.627939,2.839042,2.622117,2.000628,1.992248,1.999820,2.023938,2.001221,2.111741,2.061479,2.146891,2.186037,2.024428,2.018269,2.071206,2.029688
6,2-WEEKLY,2. ONLY Zwigato,2.002891,2.000000,2.000000,2.086728,2.045505,2.116579,2.059130,2.391302,2.290820,2.077557,2.050455,2.070137,2.058998,2.011635,2.000000,2.000000,2.096421,2.045234,2.123512,2.067461,2.437123,2.334856,2.111847,2.053618,2.093030,2.064522,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
7,2-WEEKLY,3. ONLY RHA,2.145463,2.000793,2.025834,2.469293,2.230211,2.694472,2.600966,2.878201,2.910341,2.694638,2.291998,2.492523,2.323887,2.144249,2.001429,2.022924,2.549526,2.254910,2.712667,2.616965,2.903931,2.926464,2.735107,2.361138,2.549727,2.402970,2.000000,2.000000,2.000000,2.000244,2.000000,2.001618,2.000027,2.013201,2.007643,2.000598,2.000000,2.000000,2.000000
8,2-WEEKLY,4. NONE,2.000000,2.000000,2.000875,2.130582,2.057878,2.130813,2.113230,2.301923,2.356041,2.081447,2.074005,2.114541,2.055633,2.000000,2.000000,2.003850,2.142840,2.073974,2.178938,2.131731,2.364132,2.355613,2.082661,2.058505,2.104364,2.066000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
9,2-WEEKLY,5. IOS,2.498860,2.002729,2.004903,2.868584,2.420630,2.927728,2.915694,2.974

In [18]:
cx_data13.to_clipboard(index=False)

In [24]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        /*
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.65 ) w13_ao_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.75 ) w13_ao_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.90 ) w13_ao_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.95 ) w13_ao_ad_p95,

        /*
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.65 ) w13_fe_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.75 ) w13_fe_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.90 ) w13_fe_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.95 ) w13_fe_ad_p95,
        
        /*
        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.65 ) w13_rr_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.75 ) w13_rr_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.90 ) w13_rr_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.95 ) w13_rr_ad_p95

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn3)
cx_data14.head()

,tech_savvy,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1. BOTH,1.305320,2.001061,2.547185,4.006766,5.001560,1.332959,2.004777,2.567742,4.004981,5.005336,1.000514,1.625030,2.010693,3.697843,4.900722
1,2. ONLY Zwigato,1.006208,1.982087,2.014149,3.415677,4.530942,1.017813,1.995972,2.017120,3.470367,4.572544,1.000000,1.061771,1.993321,3.000076,4.040995
2,3. ONLY RHA,1.014159,1.997281,2.026423,3.714041,4.869964,1.033262,1.994196,2.062639,3.696853,4.889515,1.000000,1.018639,1.907970,2.993442,4.025806
3,4. NONE,1.004258,1.907453,2.000000,3.156881,4.123394,1.011501,1.907683,2.000212,3.101744,4.188316,1.000000,1.001686,1.674471,2.842464,3.893430
4,5. IOS,1.653745,2.029842,2.873296,4.162227,5.073138,1.735111,2.043327,2.941418,4.283150,5.085378,1.043932,1.972804,2.343630,4.014167,5.000726


In [25]:
cx_data14

,tech_savvy,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,1. BOTH,1.305320,2.001061,2.547185,4.006766,5.001560,1.332959,2.004777,2.567742,4.004981,5.005336,1.000514,1.625030,2.010693,3.697843,4.900722
1,2. ONLY Zwigato,1.006208,1.982087,2.014149,3.415677,4.530942,1.017813,1.995972,2.017120,3.470367,4.572544,1.000000,1.061771,1.993321,3.000076,4.040995
2,3. ONLY RHA,1.014159,1.997281,2.026423,3.714041,4.869964,1.033262,1.994196,2.062639,3.696853,4.889515,1.000000,1.018639,1.907970,2.993442,4.025806
3,4. NONE,1.004258,1.907453,2.000000,3.156881,4.123394,1.011501,1.907683,2.000212,3.101744,4.188316,1.000000,1.001686,1.674471,2.842464,3.893430
4,5. IOS,1.653745,2.029842,2.873296,4.162227,5.073138,1.735111,2.043327,2.941418,4.283150,5.085378,1.043932,1.972804,2.343630,4.014167,5.000726


In [29]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [19]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        taxi_regularity_segment regularity_segment,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        /*
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.65 ) w13_ao_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.75 ) w13_ao_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.90 ) w13_ao_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.95 ) w13_ao_ad_p95,

        /*
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.65 ) w13_fe_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.75 ) w13_fe_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.90 ) w13_fe_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.95 ) w13_fe_ad_p95,
        
        /*
        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        */
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.65 ) w13_rr_ad_p65,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.75 ) w13_rr_ad_p75,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.90 ) w13_rr_ad_p90,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.95 ) w13_rr_ad_p95

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
'''
cx_data1 = pd.read_sql(cx_data, conn3)
cx_data1.head()

,regularity_segment,w01_ao_ad_p50,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,RARE_NEED,1.000000,1.000000,1.000000,1.000000,2.000000,2.142980,1.000000,1.000000,1.000000,2.000000,2.218703,0.000021,0.487717,0.999977,1.000000,1.969305
1,WEEKLY,1.794337,1.974660,2.004424,2.762673,3.759364,4.001435,1.969673,2.016122,2.788604,3.779983,4.034974,1.209816,1.999011,2.023085,3.000732,3.962939
2,UNKNOWN,1.000000,1.000000,1.000453,1.517344,2.000000,2.996158,1.000000,1.000609,1.582518,2.011586,2.993971,0.999999,1.000000,1.000000,1.500188,2.000000
3,DAILY,3.831203,3.983699,4.808739,5.034957,6.002546,6.981869,3.983843,4.864018,5.024013,6.009493,6.998856,3.695485,4.290003,4.999881,6.000000,6.719358
4,MONTHLY,1.000000,1.000000,1.052589,1.936886,2.437518,3.025427,1.000000,1.068949,1.944926,2.505678,3.024758,1.000000,1.000000,1.000000,2.000000,2.729294


In [26]:
cx_data1

,regularity_segment,w01_ao_ad_p50,w13_ao_ad_p50,w13_ao_ad_p65,w13_ao_ad_p75,w13_ao_ad_p90,w13_ao_ad_p95,w13_fe_ad_p50,w13_fe_ad_p65,w13_fe_ad_p75,w13_fe_ad_p90,w13_fe_ad_p95,w13_rr_ad_p50,w13_rr_ad_p65,w13_rr_ad_p75,w13_rr_ad_p90,w13_rr_ad_p95
0,RARE_NEED,1.000000,1.000000,1.000000,1.000000,2.000000,2.142980,1.000000,1.000000,1.000000,2.000000,2.218703,0.000021,0.487717,0.999977,1.000000,1.969305
1,WEEKLY,1.794337,1.974660,2.004424,2.762673,3.759364,4.001435,1.969673,2.016122,2.788604,3.779983,4.034974,1.209816,1.999011,2.023085,3.000732,3.962939
2,UNKNOWN,1.000000,1.000000,1.000453,1.517344,2.000000,2.996158,1.000000,1.000609,1.582518,2.011586,2.993971,0.999999,1.000000,1.000000,1.500188,2.000000
3,DAILY,3.831203,3.983699,4.808739,5.034957,6.002546,6.981869,3.983843,4.864018,5.024013,6.009493,6.998856,3.695485,4.290003,4.999881,6.000000,6.719358
4,MONTHLY,1.000000,1.000000,1.052589,1.936886,2.437518,3.025427,1.000000,1.068949,1.944926,2.505678,3.024758,1.000000,1.000000,1.000000,2.000000,2.729294
5,BI_WEEKLY,1.004402,1.005807,1.897127,2.000000,3.000148,3.967461,1.005896,1.927566,2.000065,2.999968,3.950076,1.000000,1.011813,1.644813,2.261951,3.031003


In [27]:
cx_data1.to_clipboard()

### ps_tag_link

In [77]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317_link
    )
    
    
    select
        ps_tag_link ps_tag_link,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn4)
cx_data2.head()

,ps_tag_link,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,NPS,2.000860,1.999846,2.000516,2.013089,2.003076,2.009096,2.008534,2.005764,2.000304,2.001252,2.001490,2.002128,2.006458,2.005881,1.999553,2.000262,2.012371,2.001820,2.008640,2.000518,2.016834,2.001789,2.006552,2.006537,2.004250,2.003325,1.000000,1.000000,1.000000,1.000519,1.000000,1.0,1.000239,1.001910,1.000469,1.000000,1.000000,1.000049,1.000000
1,PS,2.062422,2.009317,2.001710,2.237283,2.118606,2.244444,2.200230,2.339870,2.369625,2.187213,2.131072,2.246406,2.244029,2.077683,2.008151,2.028598,2.277092,2.113607,2.290691,2.217142,2.369338,2.446742,2.228312,2.210967,2.333848,2.208524,0.998512,0.763162,0.896235,0.995292,0.996489,1.0,0.990055,0.999105,1.000000,0.978103,0.998928,0.999718,0.985544
2,UNKNOWN,1.004079,1.008003,1.008999,1.032868,1.022194,1.047671,1.035393,1.065923,1.047507,1.062116,1.039074,1.081227,1.057120,1.015275,1.009936,1.011415,1.067522,1.020040,1.058099,1.044062,1.103778,1.075522,1.048177,1.080657,1.104080,1.135996,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [81]:
cx_data2

,ps_tag_link,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,NPS,2.000860,1.999846,2.000516,2.013089,2.003076,2.009096,2.008534,2.005764,2.000304,2.001252,2.001490,2.002128,2.006458,2.005881,1.999553,2.000262,2.012371,2.001820,2.008640,2.000518,2.016834,2.001789,2.006552,2.006537,2.004250,2.003325,1.000000,1.000000,1.000000,1.000519,1.000000,1.0,1.000239,1.001910,1.000469,1.000000,1.000000,1.000049,1.000000
1,PS,2.062422,2.009317,2.001710,2.237283,2.118606,2.244444,2.200230,2.339870,2.369625,2.187213,2.131072,2.246406,2.244029,2.077683,2.008151,2.028598,2.277092,2.113607,2.290691,2.217142,2.369338,2.446742,2.228312,2.210967,2.333848,2.208524,0.998512,0.763162,0.896235,0.995292,0.996489,1.0,0.990055,0.999105,1.000000,0.978103,0.998928,0.999718,0.985544
2,UNKNOWN,1.004079,1.008003,1.008999,1.032868,1.022194,1.047671,1.035393,1.065923,1.047507,1.062116,1.039074,1.081227,1.057120,1.015275,1.009936,1.011415,1.067522,1.020040,1.058099,1.044062,1.103778,1.075522,1.048177,1.080657,1.104080,1.135996,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [94]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317_auto
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn4)
cx_data3.head()

,ps_tag_auto,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,UNKNOWN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000006,1.000027,1.000000,1.000000,1.000169,1.000265,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000158,1.000000,1.000021,1.000095,1.000133,0.0,0.000005,0.0,0.0,0.000013,0.000627,0.003255,0.021461,0.032818,0.018203,0.027881,0.048104,0.040823
1,PS,1.998326,1.969531,1.980071,1.998694,1.999484,1.997308,1.990115,1.995183,1.996360,1.992042,1.991764,1.999952,1.999089,1.998803,1.974963,1.979717,1.999472,1.998385,1.999485,1.999461,1.997180,1.996834,1.998587,1.997412,1.999182,1.999817,1.0,1.000000,1.0,1.0,1.000000,0.999634,0.999476,1.000000,1.000000,0.999455,0.999065,0.999886,1.000000
2,NPS,1.989130,1.909854,1.911929,1.981853,1.972923,1.999779,1.986549,1.989282,1.998012,1.983348,1.989270,1.996149,1.973321,1.988112,1.896438,1.970729,1.992290,1.980594,1.998111,1.999462,1.983544,1.998605,1.995078,1.980900,1.987629,1.988430,1.0,0.999998,1.0,1.0,1.000000,1.000014,1.000000,1.000000,1.000114,1.000000,1.000000,1.000000,1.000167


In [95]:
cx_data3

,ps_tag_auto,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,UNKNOWN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000006,1.000027,1.000000,1.000000,1.000169,1.000265,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000158,1.000000,1.000021,1.000095,1.000133,0.0,0.000005,0.0,0.0,0.000013,0.000627,0.003255,0.021461,0.032818,0.018203,0.027881,0.048104,0.040823
1,PS,1.998326,1.969531,1.980071,1.998694,1.999484,1.997308,1.990115,1.995183,1.996360,1.992042,1.991764,1.999952,1.999089,1.998803,1.974963,1.979717,1.999472,1.998385,1.999485,1.999461,1.997180,1.996834,1.998587,1.997412,1.999182,1.999817,1.0,1.000000,1.0,1.0,1.000000,0.999634,0.999476,1.000000,1.000000,0.999455,0.999065,0.999886,1.000000
2,NPS,1.989130,1.909854,1.911929,1.981853,1.972923,1.999779,1.986549,1.989282,1.998012,1.983348,1.989270,1.996149,1.973321,1.988112,1.896438,1.970729,1.992290,1.980594,1.998111,1.999462,1.983544,1.998605,1.995078,1.980900,1.987629,1.988430,1.0,0.999998,1.0,1.0,1.000000,1.000014,1.000000,1.000000,1.000114,1.000000,1.000000,1.000000,1.000167


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [49]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn3)
cx_data4.head()

,taxi_income_segment,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,LOW_INCOME,1.001588,1.000000,1.000000,1.000709,1.009078,1.009574,1.009405,1.018881,1.030157,1.005870,1.008056,1.024184,1.028013,1.009276,1.000000,1.000000,1.007380,1.010381,1.005266,1.002937,1.025681,1.027280,1.027951,1.014304,1.027474,1.041912,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,UNKNOWN,1.025105,1.008198,1.021248,1.141507,1.102311,1.097258,1.139585,1.150744,1.192201,1.103838,1.113421,1.197152,1.148439,1.035696,1.016723,1.002510,1.144000,1.136545,1.122366,1.084150,1.184807,1.220131,1.168827,1.165816,1.177676,1.223537,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000014,1.000000,1.000000,1.000000,1.000000,1.000000
2,MEDIUM_INCOME,1.014714,1.007363,1.002389,1.005334,1.007403,1.022989,1.013573,1.023562,1.068432,1.019838,1.021555,1.036470,1.070734,1.010900,1.009008,1.000292,1.013688,1.016215,1.043295,1.015073,1.028697,1.075512,1.035254,1.015451,1.082101,1.036875,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,HIGH_INCOME,1.188548,1.016813,1.038889,1.230904,1.136966,1.220555,1.163300,1.252561,1.307349,1.219796,1.243180,1.272580,1.266885,1.197679,1.027277,1.061859,1.222384,1.196278,1.249291,1.219645,1.310333,1.315964,1.294572,1.277308,1.328864,1.315259,1.000014,1.0,1.0,1.0,1.0,1.000297,1.0,1.000407,1.000237,1.000159,1.000077,1.000033,1.000032


In [50]:
cx_data4

,taxi_income_segment,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,LOW_INCOME,1.001588,1.000000,1.000000,1.000709,1.009078,1.009574,1.009405,1.018881,1.030157,1.005870,1.008056,1.024184,1.028013,1.009276,1.000000,1.000000,1.007380,1.010381,1.005266,1.002937,1.025681,1.027280,1.027951,1.014304,1.027474,1.041912,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,UNKNOWN,1.025105,1.008198,1.021248,1.141507,1.102311,1.097258,1.139585,1.150744,1.192201,1.103838,1.113421,1.197152,1.148439,1.035696,1.016723,1.002510,1.144000,1.136545,1.122366,1.084150,1.184807,1.220131,1.168827,1.165816,1.177676,1.223537,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000014,1.000000,1.000000,1.000000,1.000000,1.000000
2,MEDIUM_INCOME,1.014714,1.007363,1.002389,1.005334,1.007403,1.022989,1.013573,1.023562,1.068432,1.019838,1.021555,1.036470,1.070734,1.010900,1.009008,1.000292,1.013688,1.016215,1.043295,1.015073,1.028697,1.075512,1.035254,1.015451,1.082101,1.036875,1.000000,1.0,1.0,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,HIGH_INCOME,1.188548,1.016813,1.038889,1.230904,1.136966,1.220555,1.163300,1.252561,1.307349,1.219796,1.243180,1.272580,1.266885,1.197679,1.027277,1.061859,1.222384,1.196278,1.249291,1.219645,1.310333,1.315964,1.294572,1.277308,1.328864,1.315259,1.000014,1.0,1.0,1.0,1.0,1.000297,1.0,1.000407,1.000237,1.000159,1.000077,1.000033,1.000032


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [29]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        taxi_service_affinity taxi_service_affinity,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn3)
cx_data5.head()

,taxi_service_affinity,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,LINK_CAB,1.014289,1.054737,1.013340,1.127796,1.007691,1.093242,1.073260,1.193124,1.466012,1.335874,1.232459,1.380503,1.712058,1.013150,1.077935,1.000000,1.212131,1.055427,1.128783,1.113157,1.208748,1.517345,1.449327,1.285909,1.525735,1.763682,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,ONLY_AUTO,2.000406,1.996221,1.991569,2.006416,2.000066,2.001842,2.001212,2.011773,2.006341,2.001760,2.000523,2.016598,2.004637,2.005807,1.993512,1.999259,2.001164,1.999997,2.001375,2.014697,2.012052,2.005787,2.016803,2.001792,2.011994,2.013457,1.871997,1.472432,1.525127,1.899373,1.860203,1.963218,1.943625,1.977814,1.973986,1.977736,1.971928,1.996314,1.980001
2,AUTO_CAB,1.991024,1.944045,1.988974,1.987520,1.982252,2.000000,1.999858,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.997729,1.952362,1.935512,2.000000,1.995302,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.127776,1.024501,1.023905,1.039313,1.111036,1.377741,1.228171,1.504775,1.647678,1.468650,1.334942,1.589742,1.468722
3,ONLY_LINK,2.000051,1.995291,1.999660,2.001373,2.000115,2.001519,2.000456,2.015225,2.017815,2.000830,2.000253,2.015486,2.001706,2.000045,1.999087,1.999658,2.006881,2.000000,2.000558,2.005011,2.002523,2.016439,2.003662,2.000884,2.003158,2.004671,1.702277,1.307376,1.497655,1.799513,1.676298,1.811862,1.745312,1.928383,1.921548,1.824810,1.788599,1.926603,1.924866
4,LINK_AUTO,1.994976,1.955666,1.973413,2.000000,2.000000,2.000000,1.999577,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.965446,1.988124,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.073891,1.011370,1.019125,1.090075,1.048755,1.273550,1.190575,1.511382,1.579033,1.300346,1.347076,1.621355,1.536954


In [53]:
cx_data5

,taxi_service_affinity,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,LINK_CAB,1.014289,1.054737,1.013340,1.127796,1.007691,1.093242,1.073260,1.193124,1.466012,1.335874,1.232459,1.380503,1.712058,1.013150,1.077935,1.000000,1.212131,1.055427,1.128783,1.113157,1.208748,1.517345,1.449327,1.285909,1.525735,1.763682,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,ONLY_AUTO,2.000406,1.996221,1.991569,2.006416,2.000066,2.001842,2.001212,2.011773,2.006341,2.001760,2.000523,2.016598,2.004637,2.005807,1.993512,1.999259,2.001164,1.999997,2.001375,2.014697,2.012052,2.005787,2.016803,2.001792,2.011994,2.013457,1.871997,1.472432,1.525127,1.899373,1.860203,1.963218,1.943625,1.977814,1.973986,1.977736,1.971928,1.996314,1.980001
2,AUTO_CAB,1.991024,1.944045,1.988974,1.987520,1.982252,2.000000,1.999858,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.997729,1.952362,1.935512,2.000000,1.995302,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.127776,1.024501,1.023905,1.039313,1.111036,1.377741,1.228171,1.504775,1.647678,1.468650,1.334942,1.589742,1.468722
3,ONLY_LINK,2.000051,1.995291,1.999660,2.001373,2.000115,2.001519,2.000456,2.015225,2.017815,2.000830,2.000253,2.015486,2.001706,2.000045,1.999087,1.999658,2.006881,2.000000,2.000558,2.005011,2.002523,2.016439,2.003662,2.000884,2.003158,2.004671,1.702277,1.307376,1.497655,1.799513,1.676298,1.811862,1.745312,1.928383,1.921548,1.824810,1.788599,1.926603,1.924866
4,LINK_AUTO,1.994976,1.955666,1.973413,2.000000,2.000000,2.000000,1.999577,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.965446,1.988124,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.073891,1.011370,1.019125,1.090075,1.048755,1.273550,1.190575,1.511382,1.579033,1.300346,1.347076,1.621355,1.536954
5,NO_AFFINITY,2.003312,2.000000,2.000000,2.010210,2.018905,2.028699,2.019847,2.107251,2.174260,2.032353,2.053150,2.071855,2.054622,2.003888,2.000000,2.000715,2.025949,2.002988,2.036432,2.017969,2.118923,2.149242,2.077870,2.067540,2.127743,2.115704,1.983535,1.853281,1.911419,1.990634,1.979081,1.987341,1.990476,1.999626,2.000000,2.000000,1.997953,1.999591,1.999548
6,ONLY_CAB,1.969643,1.967256,1.862947,1.995937,1.978709,2.000000,1.994299,2.000000,2.000000,2.000000,1.991308,2.000000,2.000000,1.974050,1.918263,1.908245,2.000000,1.985579,2.000000,1.997991,2.000000,2.000000,2.000000,1.996790,2.000000,2.000000,1.030987,1.005262,1.000000,1.133815,1.032883,1.380854,1.172015,1.384480,1.468182,1.276604,1.150661,1.356598,1.293176
7,UNKNOWN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.999956,0.999255,1.000000,0.999601,0.999900,0.999981,1.000000,1.000000,0.999960,1.000000,1.000000,1.000000


### RHA Check

In [37]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn3)
cx_data6.head()

,RHA_Check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,NRHA,1.000000,1.000328,1.000317,1.001154,1.001537,1.003531,1.000443,1.002528,1.000559,1.002019,1.004224,1.000374,1.000915,1.002400,1.000034,1.000708,1.001473,1.004867,1.004380,1.002465,1.002482,1.002463,1.003899,1.000325,1.002396,1.002837,1.000000,1.000000,1.0,1.000000,1.000000,1.00000,1.00000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000
1,RHA,1.082952,1.025423,1.024599,1.182989,1.054022,1.132304,1.187202,1.214552,1.208396,1.162923,1.190722,1.228914,1.181038,1.097008,1.016438,1.051313,1.132150,1.106040,1.173591,1.129491,1.223920,1.235365,1.175934,1.219613,1.227055,1.262880,1.000000,1.000000,1.0,1.000000,1.000000,1.00000,1.00000,1.00000,1.000043,1.000041,1.000000,1.000046,1.000011
2,UNKNOWN,1.411783,1.107478,1.161462,1.585744,1.419247,1.618760,1.583765,1.666432,1.738126,1.626434,1.598784,1.693399,1.663455,1.469989,1.202782,1.182992,1.628044,1.467363,1.695974,1.609101,1.690201,1.771977,1.668252,1.670729,1.781858,1.704773,1.002882,1.000004,1.0,1.000663,1.000304,1.00483,1.00668,1.02575,1.032962,1.000581,1.026846,1.039781,1.007415


In [55]:
cx_data6

,RHA_Check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,NRHA,1.000000,1.000328,1.000317,1.001154,1.001537,1.003531,1.000443,1.002528,1.000559,1.002019,1.004224,1.000374,1.000915,1.002400,1.000034,1.000708,1.001473,1.004867,1.004380,1.002465,1.002482,1.002463,1.003899,1.000325,1.002396,1.002837,1.000000,1.000000,1.0,1.000000,1.000000,1.00000,1.00000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000
1,RHA,1.082952,1.025423,1.024599,1.182989,1.054022,1.132304,1.187202,1.214552,1.208396,1.162923,1.190722,1.228914,1.181038,1.097008,1.016438,1.051313,1.132150,1.106040,1.173591,1.129491,1.223920,1.235365,1.175934,1.219613,1.227055,1.262880,1.000000,1.000000,1.0,1.000000,1.000000,1.00000,1.00000,1.00000,1.000043,1.000041,1.000000,1.000046,1.000011
2,UNKNOWN,1.411783,1.107478,1.161462,1.585744,1.419247,1.618760,1.583765,1.666432,1.738126,1.626434,1.598784,1.693399,1.663455,1.469989,1.202782,1.182992,1.628044,1.467363,1.695974,1.609101,1.690201,1.771977,1.668252,1.670729,1.781858,1.704773,1.002882,1.000004,1.0,1.000663,1.000304,1.00483,1.00668,1.02575,1.032962,1.000581,1.026846,1.039781,1.007415


### Zwigato_check

In [41]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn3)
cx_data7.head()

,Zwigato_check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,Zwigato,1.095055,1.032231,1.025435,1.104689,1.066423,1.167342,1.174629,1.198743,1.263951,1.191964,1.143614,1.277585,1.218711,1.085309,1.038573,1.041691,1.159222,1.113064,1.192116,1.168807,1.247219,1.286253,1.200638,1.225317,1.256062,1.280145,1.000000,1.0,1.000000,1.000000,1.0,1.00000,1.000000,1.000000,1.000199,1.000000,1.000017,1.000045,1.000053
1,Non-Zwigato,1.001809,1.000333,1.001841,1.003257,1.003611,1.006799,1.005894,1.002855,1.005189,1.009835,1.002837,1.010487,1.012579,1.003038,1.001635,1.003316,1.001834,1.000257,1.004204,1.007977,1.001634,1.002122,1.011130,1.002620,1.011773,1.014160,1.000000,1.0,1.000000,1.000000,1.0,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,UNKNOWN,1.424545,1.172341,1.123293,1.553647,1.423843,1.620225,1.575302,1.698356,1.691277,1.622018,1.629169,1.734503,1.686425,1.476265,1.220768,1.206996,1.635465,1.484975,1.666071,1.641166,1.689560,1.718569,1.681706,1.651906,1.730544,1.739677,1.001652,1.0,1.000091,1.002054,1.0,1.00605,1.001364,1.026602,1.021899,1.013701,1.022625,1.020620,1.024776


In [57]:
cx_data7

,Zwigato_check,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,Zwigato,1.095055,1.032231,1.025435,1.104689,1.066423,1.167342,1.174629,1.198743,1.263951,1.191964,1.143614,1.277585,1.218711,1.085309,1.038573,1.041691,1.159222,1.113064,1.192116,1.168807,1.247219,1.286253,1.200638,1.225317,1.256062,1.280145,1.000000,1.0,1.000000,1.000000,1.0,1.00000,1.000000,1.000000,1.000199,1.000000,1.000017,1.000045,1.000053
1,Non-Zwigato,1.001809,1.000333,1.001841,1.003257,1.003611,1.006799,1.005894,1.002855,1.005189,1.009835,1.002837,1.010487,1.012579,1.003038,1.001635,1.003316,1.001834,1.000257,1.004204,1.007977,1.001634,1.002122,1.011130,1.002620,1.011773,1.014160,1.000000,1.0,1.000000,1.000000,1.0,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,UNKNOWN,1.424545,1.172341,1.123293,1.553647,1.423843,1.620225,1.575302,1.698356,1.691277,1.622018,1.629169,1.734503,1.686425,1.476265,1.220768,1.206996,1.635465,1.484975,1.666071,1.641166,1.689560,1.718569,1.681706,1.651906,1.730544,1.739677,1.001652,1.0,1.000091,1.002054,1.0,1.00605,1.001364,1.026602,1.021899,1.013701,1.022625,1.020620,1.024776


### Office Usecases

In [59]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn3)
cx_data8.head()

,office_tag,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,UNKNOWN,1.399938,1.171468,1.125516,1.583642,1.419287,1.640633,1.564115,1.679016,1.679943,1.634938,1.601653,1.684824,1.674354,1.454178,1.220720,1.219955,1.645925,1.465035,1.704973,1.621514,1.717778,1.791713,1.679021,1.654728,1.733645,1.704492,1.000303,1.0,1.0,1.000846,1.005963,1.002716,1.003528,1.022284,1.03789,1.01066,1.008869,1.035192,1.018252
1,no-office-apps,1.034403,1.001017,1.011334,1.050396,1.016129,1.067816,1.023668,1.083904,1.081293,1.058803,1.042402,1.045557,1.061098,1.036246,1.009599,1.003293,1.058561,1.025564,1.047581,1.068118,1.126216,1.104157,1.052780,1.053519,1.040155,1.081899,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000
2,office-apps,1.117652,1.010056,1.012575,1.156538,1.041719,1.196386,1.098539,1.231046,1.258020,1.201284,1.135507,1.236074,1.228496,1.097218,1.004965,1.015356,1.174101,1.075680,1.200868,1.240274,1.233423,1.281824,1.193399,1.257233,1.276889,1.223311,1.000000,1.0,1.0,1.000000,1.000000,1.000094,1.000002,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000


In [71]:
cx_data8

,office_tag,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,UNKNOWN,1.399938,1.171468,1.125516,1.583642,1.419287,1.640633,1.564115,1.679016,1.679943,1.634938,1.601653,1.684824,1.674354,1.454178,1.220720,1.219955,1.645925,1.465035,1.704973,1.621514,1.717778,1.791713,1.679021,1.654728,1.733645,1.704492,1.000303,1.0,1.0,1.000846,1.005963,1.002716,1.003528,1.022284,1.03789,1.01066,1.008869,1.035192,1.018252
1,no-office-apps,1.034403,1.001017,1.011334,1.050396,1.016129,1.067816,1.023668,1.083904,1.081293,1.058803,1.042402,1.045557,1.061098,1.036246,1.009599,1.003293,1.058561,1.025564,1.047581,1.068118,1.126216,1.104157,1.052780,1.053519,1.040155,1.081899,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000
2,office-apps,1.117652,1.010056,1.012575,1.156538,1.041719,1.196386,1.098539,1.231046,1.258020,1.201284,1.135507,1.236074,1.228496,1.097218,1.004965,1.015356,1.174101,1.075680,1.200868,1.240274,1.233423,1.281824,1.193399,1.257233,1.276889,1.223311,1.000000,1.0,1.0,1.000000,1.000000,1.000094,1.000002,1.000000,1.00000,1.00000,1.000000,1.000000,1.000000


### Age Group

In [67]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        approx_percentile(case when week_start_date = '20241216' then ao_day end, 0.5) w01_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then ao_day end, 0.5 ) w02_ao_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then ao_day end, 0.5 ) w03_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then ao_day end, 0.5 ) w04_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then ao_day end, 0.5 ) w05_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then ao_day end, 0.5 ) w06_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then ao_day end, 0.5 ) w07_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then ao_day end, 0.5 ) w08_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then ao_day end, 0.5 ) w09_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then ao_day end, 0.5 ) w10_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then ao_day end, 0.5 ) w11_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then ao_day end, 0.5 ) w12_ao_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then ao_day end, 0.5 ) w13_ao_ad_p50,
        
        approx_percentile(case when week_start_date = '20241216' then fe_day end, 0.5 ) w01_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then fe_day end, 0.5 ) w02_fe_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then fe_day end, 0.5 ) w03_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then fe_day end, 0.5 ) w04_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then fe_day end, 0.5 ) w05_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then fe_day end, 0.5 ) w06_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then fe_day end, 0.5 ) w07_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then fe_day end, 0.5 ) w08_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then fe_day end, 0.5 ) w09_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then fe_day end, 0.5 ) w10_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then fe_day end, 0.5 ) w11_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then fe_day end, 0.5 ) w12_fe_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then fe_day end, 0.5 ) w13_fe_ad_p50,

        approx_percentile(case when week_start_date = '20241216' then rr_day end, 0.5 ) w01_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241223' then rr_day end, 0.5 ) w02_rr_ad_p50,
        approx_percentile(case when week_start_date = '20241230' then rr_day end, 0.5 ) w03_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250106' then rr_day end, 0.5 ) w04_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250113' then rr_day end, 0.5 ) w05_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250120' then rr_day end, 0.5 ) w06_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250127' then rr_day end, 0.5 ) w07_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250203' then rr_day end, 0.5 ) w08_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250210' then rr_day end, 0.5 ) w09_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250217' then rr_day end, 0.5 ) w10_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250224' then rr_day end, 0.5 ) w11_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250303' then rr_day end, 0.5 ) w12_rr_ad_p50,
        approx_percentile(case when week_start_date = '20250310' then rr_day end, 0.5 ) w13_rr_ad_p50

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn3)
cx_data9.head(10)

,age_group,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,21-25,1.004856,1.001578,1.001433,1.004129,1.018122,1.033039,1.008040,1.066769,1.099804,1.071892,1.060607,1.089632,1.074572,1.011526,1.001998,1.005839,1.007980,1.031567,1.017405,1.016912,1.113704,1.116208,1.096381,1.054065,1.068291,1.134863,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
1,26-30,1.037219,1.013851,1.005585,1.051335,1.031581,1.050981,1.066632,1.168898,1.225346,1.141996,1.127118,1.115772,1.110635,1.041999,1.013897,1.010538,1.089761,1.042445,1.103907,1.060927,1.159664,1.166594,1.175840,1.179709,1.236276,1.234025,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
2,31-35,1.028966,1.000781,1.004668,1.047780,1.032601,1.075952,1.041052,1.058584,1.201707,1.098087,1.064715,1.146949,1.081750,1.027561,1.003188,1.004027,1.059783,1.038056,1.062453,1.068311,1.158246,1.147269,1.122092,1.144387,1.178005,1.114355,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
3,36-45,1.019818,1.002469,1.001388,1.042677,1.021480,1.083358,1.049538,1.108418,1.101923,1.107161,1.068096,1.067839,1.073674,1.019963,1.010882,1.000690,1.064831,1.037258,1.096017,1.057779,1.129814,1.172295,1.132593,1.096796,1.096089,1.095168,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000063,1.0
4,Above-45,1.028614,1.000123,1.000000,1.022881,1.016232,1.039116,1.017742,1.049060,1.050550,1.041746,1.045066,1.111087,1.048919,1.024197,1.000666,1.000135,1.032919,1.022048,1.037022,1.023557,1.066370,1.069778,1.058867,1.049083,1.064098,1.045496,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
5,Below-21,1.002899,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.047722,1.038084,1.023783,1.009742,1.022326,1.032923,1.000000,1.000000,1.000000,1.013073,1.000000,1.004538,1.000000,1.059130,1.042242,1.029148,1.020730,1.035398,1.032303,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
6,UNKNOWN,1.400181,1.151801,1.105716,1.537444,1.376218,1.597867,1.546732,1.413193,1.301320,1.229891,1.183075,1.286506,1.318889,1.419715,1.149126,1.135557,1.597197,1.442685,1.671528,1.571480,1.473266,1.331367,1.274915,1.239075,1.335552,1.316765,1.000061,1.0,1.000027,1.000794,1.000387,1.018441,1.00081,1.000512,1.0,1.0,1.0,1.000000,1.0


In [73]:
cx_data9

,age_group,w01_ao_ad_p50,w02_ao_ad_p50,w03_ao_ad_p50,w04_ao_ad_p50,w05_ao_ad_p50,w06_ao_ad_p50,w07_ao_ad_p50,w08_ao_ad_p50,w09_ao_ad_p50,w10_ao_ad_p50,w11_ao_ad_p50,w12_ao_ad_p50,w13_ao_ad_p50,w01_fe_ad_p50,w02_fe_ad_p50,w03_fe_ad_p50,w04_fe_ad_p50,w05_fe_ad_p50,w06_fe_ad_p50,w07_fe_ad_p50,w08_fe_ad_p50,w09_fe_ad_p50,w10_fe_ad_p50,w11_fe_ad_p50,w12_fe_ad_p50,w13_fe_ad_p50,w01_rr_ad_p50,w02_rr_ad_p50,w03_rr_ad_p50,w04_rr_ad_p50,w05_rr_ad_p50,w06_rr_ad_p50,w07_rr_ad_p50,w08_rr_ad_p50,w09_rr_ad_p50,w10_rr_ad_p50,w11_rr_ad_p50,w12_rr_ad_p50,w13_rr_ad_p50
0,21-25,1.004856,1.001578,1.001433,1.004129,1.018122,1.033039,1.008040,1.066769,1.099804,1.071892,1.060607,1.089632,1.074572,1.011526,1.001998,1.005839,1.007980,1.031567,1.017405,1.016912,1.113704,1.116208,1.096381,1.054065,1.068291,1.134863,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
1,26-30,1.037219,1.013851,1.005585,1.051335,1.031581,1.050981,1.066632,1.168898,1.225346,1.141996,1.127118,1.115772,1.110635,1.041999,1.013897,1.010538,1.089761,1.042445,1.103907,1.060927,1.159664,1.166594,1.175840,1.179709,1.236276,1.234025,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
2,31-35,1.028966,1.000781,1.004668,1.047780,1.032601,1.075952,1.041052,1.058584,1.201707,1.098087,1.064715,1.146949,1.081750,1.027561,1.003188,1.004027,1.059783,1.038056,1.062453,1.068311,1.158246,1.147269,1.122092,1.144387,1.178005,1.114355,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
3,36-45,1.019818,1.002469,1.001388,1.042677,1.021480,1.083358,1.049538,1.108418,1.101923,1.107161,1.068096,1.067839,1.073674,1.019963,1.010882,1.000690,1.064831,1.037258,1.096017,1.057779,1.129814,1.172295,1.132593,1.096796,1.096089,1.095168,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000063,1.0
4,Above-45,1.028614,1.000123,1.000000,1.022881,1.016232,1.039116,1.017742,1.049060,1.050550,1.041746,1.045066,1.111087,1.048919,1.024197,1.000666,1.000135,1.032919,1.022048,1.037022,1.023557,1.066370,1.069778,1.058867,1.049083,1.064098,1.045496,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
5,Below-21,1.002899,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.047722,1.038084,1.023783,1.009742,1.022326,1.032923,1.000000,1.000000,1.000000,1.013073,1.000000,1.004538,1.000000,1.059130,1.042242,1.029148,1.020730,1.035398,1.032303,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.0,1.0,1.0,1.000000,1.0
6,UNKNOWN,1.400181,1.151801,1.105716,1.537444,1.376218,1.597867,1.546732,1.413193,1.301320,1.229891,1.183075,1.286506,1.318889,1.419715,1.149126,1.135557,1.597197,1.442685,1.671528,1.571480,1.473266,1.331367,1.274915,1.239075,1.335552,1.316765,1.000061,1.0,1.000027,1.000794,1.000387,1.018441,1.00081,1.000512,1.0,1.0,1.0,1.000000,1.0


### Office Use-Cases

In [61]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73


In [62]:
cx_data11

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73
